# Assignment 03 — development error review and final freeze

## tl;dr

All 4,203 development feature records and all six fitted-model hashes reproduce the original experiment exactly. RF hybrid F1 drops 7.21 percentage points under complexity-block shuffle and 5.72 points within password/language/shard cells. The password-present F1 gap has an exploratory group-bootstrap interval of −15.53 to +1.69 points. Keep the frozen internal benchmark with its limitations; test stays sealed.

## Context & Methods

The review reproduces the exact 4,203 development feature rows and six baseline models. Complexity bins use train tag-count quartiles. Password gaps use whole-group bootstrap; standardization is descriptive. Permutation means measure input reliance under artificial feature shuffles, not adversarial robustness.

### Key Assumptions

Published source labels are not independently adjudicated. All shards belong to one historical source cohort. The 30 shuffle repeats quantify shuffle variability; they are not a population confidence interval. No feature, threshold, model or seed is selected from these diagnostics.

## Data

Run from the repository or its notebooks directory. Only committed sanitized evidence is read; no live page, raw HTML or test feature matrix is accessed.

In [1]:
from pathlib import Path
import json, hashlib, sys
root = Path.cwd()
if not (root / 'config').exists():
    root = root.parent
assert (root / 'config/assignment03_final_evaluation_v1.json').exists()
sys.path.insert(0, str(root / 'src'))
from phishing_url.final_evaluation_contract import verify
freeze_check = verify(root)
print(freeze_check)
evidence = root / 'results/assignment03/review_v1'
load = lambda name: json.loads((evidence / name).read_text())
summary = load('review_summary.json')
assert summary['test_feature_rows'] == summary['test_predictions'] == 0
print({k: summary[k] for k in ['replayed_fits', 'maximum_score_difference', 'feature_hashes_match_original', 'validation_rows', 'validation_groups']})

{'status': 'final_specification_verified_test_sealed', 'freeze_sha256': 'be19ad2c728704d75afd2eb605e023b9edfb8544d7700735e6fe071e906763e0', 'locked_files': 75, 'evaluation_cells': 6, 'fit_rows': 3461, 'future_test_rows': 742, 'test_features_read': 0, 'test_predictions': 0, 'test_execution_authorized': False, 'performance_or_deployment_approval': False}
{'replayed_fits': 6, 'maximum_score_difference': 0.0, 'feature_hashes_match_original': True, 'validation_rows': 742, 'validation_groups': 556}


## Results

### Password-present and password-absent pages

In [2]:
condition = 'random_forest/url_dom/baseline'
for row in load('slices.json'):
    if row['condition'] == condition and row['facet'] == 'password_present':
        print({k: row[k] for k in ['value','n','positive_n','negative_n','fp','fn','f1','recall','fpr']})
print('Present minus absent, exploratory group-bootstrap intervals:')
print(load('password_gap_intervals.json')[condition])
print('Complexity-standardized rates and retained support:')
print(load('password_complexity_standardization.json')[condition])

{'value': 'False', 'n': 597, 'positive_n': 288.0, 'negative_n': 309.0, 'fp': 19.0, 'fn': 24.0, 'f1': 0.9246935201401051, 'recall': 0.9166666666666666, 'fpr': 0.061488673139158574}
{'value': 'True', 'n': 145, 'positive_n': 85.0, 'negative_n': 60.0, 'fp': 7.0, 'fn': 14.0, 'f1': 0.8711656441717791, 'recall': 0.8352941176470589, 'fpr': 0.11666666666666667}
Present minus absent, exploratory group-bootstrap intervals:
{'f1': {'high': 0.01692585618350054, 'low': -0.15528997526788116, 'valid_replicates': 2000}, 'fpr': {'high': 0.14631555897823253, 'low': -0.01901225790682516, 'valid_replicates': 2000}, 'recall': {'high': 0.026271530950384724, 'low': -0.2287700096400201, 'valid_replicates': 2000}}
Complexity-standardized rates and retained support:
{'fnr': {'absent': 0.09173403332640766, 'available': True, 'bins': [{'absent': 134, 'bin': 0, 'present': 19, 'weight': 0.4101876675603217}, {'absent': 76, 'bin': 1, 'present': 24, 'weight': 0.2680965147453083}, {'absent': 58, 'bin': 2, 'present': 30,

### Complexity and feature reliance

In [3]:
print(load('complexity_distributions.json'))
for row in load('permutation_summary.json'):
    if row['condition'] == condition:
        print(row['mode'], row['block'], 'mean F1 loss (percentage points):', round(100 * row['f1_drop']['mean'], 3), 'shuffle SD:', round(100 * row['f1_drop']['std_sample'], 3), 'exchangeable rows:', row['eligible_rows'])

{'boundaries': 'searchsorted right: x < cut0, cut0 <= x < cut1, etc.; ties may make bins unequal', 'cuts': [106.0, 475.0, 1083.0], 'rows': [{'benign': 125, 'bin': 0, 'n': 860, 'partition': 'train', 'password_present': 157, 'phishing': 735}, {'benign': 348, 'bin': 1, 'n': 870, 'partition': 'train', 'password_present': 258, 'phishing': 522}, {'benign': 548, 'bin': 2, 'n': 864, 'partition': 'train', 'password_present': 118, 'phishing': 316}, {'benign': 697, 'bin': 3, 'n': 867, 'partition': 'train', 'password_present': 102, 'phishing': 170}, {'benign': 30, 'bin': 0, 'n': 183, 'partition': 'validation', 'password_present': 35, 'phishing': 153}, {'benign': 69, 'bin': 1, 'n': 169, 'partition': 'validation', 'password_present': 46, 'phishing': 100}, {'benign': 128, 'bin': 2, 'n': 216, 'partition': 'validation', 'password_present': 39, 'phishing': 88}, {'benign': 142, 'bin': 3, 'n': 174, 'partition': 'validation', 'password_present': 25, 'phishing': 32}]}
unconditional all_dom mean F1 loss (per

### Hybrid changes relative to each comparator

In [4]:
for row in load('paired_error_cohorts.json'):
    if row['model'] == 'random_forest' and row['kind'] in ['fixes', 'breaks']:
        print(row)
assert load('independent_verification.json')['verification_passed']
print('All reference comparisons and independent evidence checks passed.')

{'benign': 29, 'comparator': 'url_only', 'kind': 'fixes', 'median_tag_count': 465.0, 'model': 'random_forest', 'n': 63, 'password_present': 22, 'phishing': 34}
{'benign': 7, 'comparator': 'url_only', 'kind': 'breaks', 'median_tag_count': 496.5, 'model': 'random_forest', 'n': 18, 'password_present': 6, 'phishing': 11}
{'benign': 25, 'comparator': 'dom_only', 'kind': 'fixes', 'median_tag_count': 178.5, 'model': 'random_forest', 'n': 46, 'password_present': 12, 'phishing': 21}
{'benign': 9, 'comparator': 'dom_only', 'kind': 'breaks', 'median_tag_count': 654.0, 'model': 'random_forest', 'n': 21, 'password_present': 8, 'phishing': 12}
All reference comparisons and independent evidence checks passed.


## Takeaways

The simplest validation pages contain 30 benign / 153 phishing examples; the most complex contain 142 / 32. RF hybrid FPR is 26.67% in the simplest bin; phishing recall is 75.00% in the most complex. Password-group error-rate gaps shrink after descriptive complexity standardization. These are cohort associations, not causal or adversarial evidence.

No correctness blocker was found. The six original cells, seed and threshold stay unchanged. The frozen specification is checked without test inference; one-time final test execution and the submission report are subsequent work.